# 05 · From LSTM to Transformer: A Fair Comparison

**Time:** about 45–60 minutes · **Prerequisites:** `01` and `02`

## Learning objectives
1. Build a Transformer-encoder text classifier in MLX: embeddings,
   sinusoidal **positional encoding**, self-attention layers, and **masked mean pooling**.
2. Run a **fair architecture comparison**: same splits, held-out data,
   several seeds, a baseline, and measured training time.
3. Explain why "Transformers are better" depends on data size and task.

## Architecture bridge
```
LSTM classifier (01–04)              Transformer classifier (this notebook)
Embedding                            Embedding + positional encoding
   |                                    |
LSTM: reads tokens one at a time     TransformerEncoder: all tokens at once
   |                                   (self-attention + feed-forward) × 2
Last real token → Linear             Mean over real tokens → Linear
```

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import mlx.core as mx
import mlx.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from mlx_nlp_utils import (
    SentimentLSTM, count_parameters, create_vocabulary, group_train_val_split,
    pad_sequences, predict_proba, print_device_info, sanity_check, texts_to_sequences,
    tokenize, train_model, train_val_split,
)

sns.set_style('whitegrid')
print_device_info()

## Part 1: Positional encoding

Self-attention treats its input as a **set**: shuffle the tokens and each
token's output is unchanged, just shuffled. To use word order, we add a
position-dependent vector to each embedding. The sinusoidal encoding from
[Vaswani et al. (2017)](https://arxiv.org/abs/1706.03762) uses sines and
cosines of different frequencies:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \qquad
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

Low dimensions oscillate quickly (fine position), high dimensions slowly
(coarse position). The table is fixed, not learned: we store it as
`_pos_encoding`, and MLX excludes underscore-prefixed attributes from the
trainable parameters.

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """Create sinusoidal positional encoding matrix."""
    pe = mx.zeros((max_len, d_model))
    position = mx.arange(0, max_len).reshape(-1, 1)
    div_term = mx.exp(mx.arange(0, d_model, 2) * (-mx.log(mx.array(10000.0)) / d_model))

    pe = pe.at[:, 0::2].add(mx.sin(position * div_term))
    pe = pe.at[:, 1::2].add(mx.cos(position * div_term[:d_model // 2]))
    return pe

pe = np.array(sinusoidal_positional_encoding(100, 64))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))
im = ax1.imshow(pe[:50], aspect='auto', cmap='RdBu')
ax1.set_xlabel('embedding dimension'); ax1.set_ylabel('position'); ax1.set_title('Positional encoding')
plt.colorbar(im, ax=ax1)
ax2.imshow(pe[:50] @ pe[:50].T, cmap='viridis')
ax2.set_xlabel('position'); ax2.set_ylabel('position'); ax2.set_title('Dot product between positions')
plt.show()

The right panel shows why this works: the dot product between two positions
depends mostly on their **distance**, so attention can learn rules like
"look one token to the left" that hold at every position.

## Part 2: The Transformer classifier

We use MLX's built-in `nn.TransformerEncoder` (notebook 07 builds the
attention layers by hand). Two details matter for padded batches:

* **Attention mask.** Real tokens must not attend to `<PAD>` positions. The
  boolean mask is `True` for allowed keys.
* **Masked mean pooling.** Average only over real tokens; including padding
  would make the representation depend on how much padding a batch has.

In [ ]:
class TransformerClassifier(nn.Module):
    """Transformer-based text classifier using MLX's TransformerEncoder."""

    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2,
                 num_classes=3, max_len=50, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self._pos_encoding = sinusoidal_positional_encoding(max_len, d_model)
        self.transformer_encoder = nn.TransformerEncoder(
            num_layers=num_layers, dims=d_model, num_heads=num_heads,
            mlp_dims=d_model * 4, dropout=dropout)
        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def __call__(self, x):
        seq_len = x.shape[1]
        h = self.embedding(x) + self._pos_encoding[:seq_len, :]

        # Boolean masks use True for allowed keys. An all-padding input gets one
        # safe key so its softmax row is not fully masked (which would be NaN).
        valid = x != 0
        safe_keys = valid | ((mx.sum(valid, axis=1, keepdims=True) == 0)
                             & (mx.arange(seq_len)[None, :] == 0))
        h = self.transformer_encoder(h, mask=safe_keys[:, None, None, :])

        weights = valid[..., None].astype(h.dtype)
        pooled = mx.sum(h * weights, axis=1) / mx.maximum(mx.sum(weights, axis=1), 1)
        return self.classifier(self.dropout(pooled))

# Padding must not change the prediction.
check = TransformerClassifier(10, d_model=16, num_heads=2, num_layers=1); check.eval()
diff = mx.abs(check(mx.array([[3, 4]])) - check(mx.array([[3, 4, 0, 0, 0]]))).max().item()
print(f"Max logit change from adding padding: {diff:.2e}")

## Part 3: Data and splits

We reuse notebook 01's lessons: the intent data is split by **base phrase**
so near-copies don't leak, sentiment is split after removing exact
duplicates, and each vocabulary is fitted on training text only.

In [ ]:
def base_phrase(text):
    phrase = " ".join(tokenize(text))
    for word in ["please", "could you", "can you", "hey", "ok"]:
        if phrase.startswith(word + " "):
            phrase = phrase[len(word) + 1:]
        if phrase.endswith(" " + word):
            phrase = phrase[: -len(word) - 1]
    return phrase

def load_task(name):
    data = json.loads(Path(f'../data/{name}_samples/data.json').read_text())
    pairs = list(dict.fromkeys(zip(data['texts'], data['labels'])))
    return pairs, sorted(set(data['labels']))

def make_split(task, pairs, seed):
    if task == 'intent':
        train, val = group_train_val_split(pairs, [base_phrase(t) for t, _ in pairs], 0.2, seed)
    else:
        train, val = train_val_split(pairs, 0.2, seed)
    return (*map(list, zip(*train)), *map(list, zip(*val)))

tasks = {name: load_task(name) for name in ('intent', 'sentiment')}
for name, (pairs, classes) in tasks.items():
    tr_t, _, va_t, _ = make_split(name, pairs, 0)
    print(f"{name}: {len(tr_t)} train / {len(va_t)} validation, classes {classes}")

## Part 4: The experiment

A comparison is only fair if both models get the same data, splits,
evaluation, and comparable tuning effort. Here:

* **Splits:** identical for both models, three seeds per task.
* **Budget:** 60 epochs, batch size 16, for both.
* **Learning rate:** 0.005 for the LSTM, 0.0005 for the Transformer. Each was
  picked from a quick manual check of two or three values; neither is tuned
  carefully. Transformers are known to need smaller learning rates (and
  often warm-up).
* **Baseline:** TF-IDF + logistic regression on the same splits.
* **Cost:** wall-clock seconds per epoch.

🤔 **Predict:** which architecture wins on ~100 training sentences?

In [ ]:
EPOCHS = 60

def build(arch, vocab_size, n_classes, max_len):
    if arch == 'LSTM':
        return SentimentLSTM(vocab_size, 64, 128, n_classes), 0.005
    return TransformerClassifier(vocab_size, d_model=64, num_heads=4, num_layers=2,
                                 num_classes=n_classes, max_len=max_len), 0.0005

def run(task, arch, seed):
    pairs, classes = tasks[task]
    tr_t, tr_l, va_t, va_l = make_split(task, pairs, seed)
    _, w2i = create_vocabulary(tr_t)
    max_len = max(len(tokenize(t)) for t in tr_t)
    enc = lambda t: mx.array(pad_sequences(texts_to_sequences(t, w2i), max_len))
    lab = lambda l: mx.array([classes.index(x) for x in l], dtype=mx.int32)
    if arch == 'TF-IDF':
        vec = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, ngram_range=(1, 2))
        clf = LogisticRegression(max_iter=1000).fit(vec.fit_transform(tr_t), tr_l)
        return dict(val_acc=float(np.mean(clf.predict(vec.transform(va_t)) == np.array(va_l))))
    mx.random.seed(seed)
    model, lr = build(arch, len(w2i), len(classes), max_len)
    start = time.perf_counter()
    model, history = train_model(model, enc(tr_t), lab(tr_l), epochs=EPOCHS, learning_rate=lr,
                                 batch_size=16, X_val=enc(va_t), y_val=lab(va_l), seed=seed)
    return dict(val_acc=history['val_accuracy'][-1], history=history, model=model,
                sec_per_epoch=(time.perf_counter() - start) / EPOCHS,
                params=count_parameters(model), w2i=w2i, max_len=max_len, classes=classes)

results = {}
for task in tasks:
    for arch in ('TF-IDF', 'LSTM', 'Transformer'):
        results[task, arch] = [run(task, arch, seed) for seed in range(3)]
        accs = [r['val_acc'] for r in results[task, arch]]
        print(f"{task:<10} {arch:<12} val acc per seed: {[f'{a:.0%}' for a in accs]}")

## Part 5: Results

In [ ]:
rows = []
print(f"{'task':<10} {'model':<12} {'mean val acc':>12} {'range':>12} {'params':>9} {'s/epoch':>8}")
for (task, arch), runs in results.items():
    accs = [r['val_acc'] for r in runs]
    params = f"{runs[0]['params']:,}" if 'params' in runs[0] else '-'
    speed = f"{np.mean([r['sec_per_epoch'] for r in runs]):.3f}" if 'sec_per_epoch' in runs[0] else '-'
    print(f"{task:<10} {arch:<12} {np.mean(accs):>12.1%} {min(accs):>5.0%}–{max(accs):<5.0%} {params:>9} {speed:>8}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, task in zip(axes, tasks):
    for i, arch in enumerate(('TF-IDF', 'LSTM', 'Transformer')):
        accs = [r['val_acc'] for r in results[task, arch]]
        ax.bar(i, np.mean(accs), color=['#bbb', '#4ECDC4', '#45B7D1'][i], alpha=0.7)
        ax.scatter([i] * len(accs), accs, color='black', zorder=3, s=20)
    ax.set_xticks(range(3)); ax.set_xticklabels(['TF-IDF', 'LSTM', 'Transformer'])
    ax.set_title(f'{task}: validation accuracy (dots = seeds)'); ax.set_ylim(0, 1.05)
plt.show()

for task in tasks:
    sanity_check(np.mean([r['val_acc'] for r in results[task, 'Transformer']]) > 1 / len(tasks[task][1]),
                 f"Transformer beats chance on {task}")

**How to read this.** Each dot is one split seed. Where the dots of two
models overlap, this experiment cannot reliably tell them apart. One model may
have a higher *mean*, but with three seeds and ~20 validation sentences, a
gap of a few points is within the seed-to-seed spread. The TF-IDF baseline
stays within reach of both neural models.

The two networks have almost the same parameter count, and their time per
epoch is similar. The Transformer's parallelism pays off with long sequences
and large batches, not with batches of 16 five-word sentences.

To *establish* a difference you would need more seeds, a larger validation
set, and a learning-rate search for each model (Exercise 1).

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, task in zip(axes, tasks):
    for arch, color in (('LSTM', '#4ECDC4'), ('Transformer', '#45B7D1')):
        h = results[task, arch][0]['history']
        ax.plot(h['loss'], color=color, label=f'{arch} train')
        ax.plot(h['val_loss'], '--', color=color, label=f'{arch} validation')
    ax.set_title(f'{task} (seed 0): loss'); ax.set_xlabel('epoch'); ax.legend(fontsize=8)
plt.show()

## Part 6: When does each architecture win?

| | LSTM | Transformer |
|---|---|---|
| How it reads | One token at a time; state carries the past | All tokens at once via attention |
| Long-range dependencies | Must survive many state updates | Any token can attend to any other directly |
| Training parallelism | Sequential over time steps | Parallel over positions (fast on long sequences, big batches) |
| Cost in sequence length | Linear | Quadratic attention matrix |
| Small data | Strong recency bias helps a little | Weak built-in bias; needs more data or pretraining |
| Measured here | see table above | see table above |

The Transformer's advantage comes with **scale**: long contexts, large
datasets, and especially **pretraining** on huge corpora (BERT, GPT). With
100 sentences and no pretraining, its flexibility is mostly unused. The
practical lesson: start from a pretrained Transformer (notebooks 08 and 10)
rather than training one from scratch on small data.

## Part 7 (optional): Repeat on SNIPS

With ~14K training queries, the picture can change. Download with
`python ../scripts/download_datasets.py --snips`; the cell below trains each
model for 3 epochs on the official split. (If the download failed and the
offline six-intent fixture was created instead, the numbers are not meaningful.)

In [ ]:
from mlx_nlp_utils import load_real_dataset

snips = load_real_dataset("snips")
if snips is not None:
    s_tr, s_trl, s_te, s_tel = snips
    s_classes = sorted(set(s_trl))
    _, s_w2i = create_vocabulary(s_tr)
    s_len = 32
    s_enc = lambda t: mx.array(pad_sequences(texts_to_sequences(t, s_w2i), s_len))
    s_lab = lambda l: mx.array([s_classes.index(x) for x in l], dtype=mx.int32)
    for arch in ('LSTM', 'Transformer'):
        mx.random.seed(0)
        model, lr = build(arch, len(s_w2i), len(s_classes), s_len)
        start = time.perf_counter()
        model, h = train_model(model, s_enc(s_tr), s_lab(s_trl), epochs=3, learning_rate=lr,
                               batch_size=64, X_val=s_enc(s_te), y_val=s_lab(s_tel))
        print(f"{arch:<12} test acc {h['val_accuracy'][-1]:.1%}, {(time.perf_counter() - start) / 3:.1f} s/epoch")

## Part 8: Try both models

In [ ]:
lstm_run, trans_run = results['sentiment', 'LSTM'][0], results['sentiment', 'Transformer'][0]
for text in ['This movie was absolutely fantastic', 'I really hated the service',
             'It was okay I guess', 'Turn on the kitchen light']:
    out = []
    for name, r in (('LSTM', lstm_run), ('Transformer', trans_run)):
        probs = predict_proba(r['model'], text, r['w2i'], r['max_len'])
        out.append(f"{name}: {r['classes'][int(probs.argmax())]} ({probs.max():.0%})")
    print(f"'{text}'\n   " + " | ".join(out))

The last sentence is not a review at all, yet both models give it a
sentiment, probably with high confidence (see notebook 02).

## Summary

* Self-attention is order-blind; positional encodings add order back.
* Padding needs both an attention mask and masked pooling.
* A fair comparison fixes the splits, runs several seeds, includes a
  baseline, and reports cost. On tiny data the architectures are statistically tied.
* Transformers win with scale and pretraining, not by default.

## 🧠 Exercises

**1. Learning-rate sensitivity.** Rerun the Transformer with learning rates
0.005 and 0.0001. 🤔 Predict which fails and how.

<details><summary>What to look for</summary>

Too high a learning rate makes training unstable (loss spikes or validation
accuracy collapses); too low leaves it underfit after 60 epochs. Comparing
architectures without giving each one its own learning-rate search is a
common way to reach the wrong conclusion.
</details>

**2. Remove positional encoding.** Replace `+ self._pos_encoding[...]` with
`+ 0` and rerun. 🤔 Predict: how much does accuracy drop on these tasks?

<details><summary>What to look for</summary>

Probably very little. Without positions (and with no causal mask), this
encoder is exactly a bag of words, and notebook 01 showed that keywords carry
most of the signal here. Notebook 07 repeats the ablation on a language model
and explains why even there the answer depends on the setting.
</details>

**3. Depth.** Try `num_layers=1` and `num_layers=4`. Does parameter count
predict validation accuracy here?